# 4.2.1 Implementing SGD in PyTorch

Notes from 2026-03-10



In [4]:
from collections.abc import Callable, Iterable
from typing import Optional
import torch
import math

class SGD(torch.optim.Optimizer):

    def __init__(self, params, lr=1e-3):
        if lr < 0:
            raise ValueError(f"Invalid learning rate: {lr}")
        defaults = {"lr": lr}
        super().__init__(params, defaults)

    def step(self, closure: Optional[Callable] = None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            lr = group["lr"] # Get the learning rate.
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p] # Get state associated with p.
                t = state.get("t", 0) # Get iteration number from the state, or initial value.
                grad = p.grad.data # Get the gradient of loss with respect to p.
                p.data -= lr / math.sqrt(t + 1) * grad # Update weight tensor in-place.
                state["t"] = t + 1 # Increment iteration number.
        return loss

In [28]:
# outer = 10 #
n_iterations = 100000
lr_list = [10, 1, 0.1, 0.01, 0.001, 0.0001, 0.00001]
lr_list = [100, 10, 1, 0.1, 0.01, 0.001]
outer = len(lr_list)

weights_0 = 5 * torch.randn((10, 10))

for i_lr, lr in enumerate(lr_list):    
    weights = weights_0.clone().detach().requires_grad_(True)
    opt = SGD([weights], lr=lr)

    print(f"learning rate {i_lr}: {lr}")
    for t in range(n_iterations):
        opt.zero_grad() # Reset the gradients for all learnable parameters.
        loss = (weights**2).mean() # Compute a scalar loss value.
        if t == 0:
            print(f"initial loss {i_lr}: {loss.cpu().item()}")
        # if t % (n_iterations//4) == 0:
        #     print(f"loss {i_lr}: {loss.cpu().item()}")
        if t == n_iterations - 1:
            print(f"final loss {i_lr}: {loss.cpu().item()}\n")
        loss.backward() # Run backward pass, which computes gradients.
        opt.step() # Run optimizer step.

learning rate 0: 100
initial loss 0: 25.187387466430664
final loss 0: 0.0

learning rate 1: 10
initial loss 1: 25.187387466430664
final loss 1: 0.0

learning rate 2: 1
initial loss 2: 25.187387466430664
final loss 2: 2.739024840270332e-10

learning rate 3: 0.1
initial loss 3: 25.187387466430664
final loss 3: 2.0184385776519775

learning rate 4: 0.01
initial loss 4: 25.187387466430664
final loss 4: 19.567686080932617

learning rate 5: 0.001
initial loss 5: 25.187387466430664
final loss 5: 24.542940139770508



We observed that large learing rates, like 100, 10 and 1 solve the problem.

The smaller learning rates, (1e-3, 1e-2), move in the right direction but don't get close to the answer in any noticeable way, even with 10k iterations.

0.1 could go either way.

Note that this example just drives the weights to zero and is a pretty easy gradient to optimize under nearly any metric.
